# 🔍 RobustLens: GPU Evaluation Pipeline

This notebook runs the heavy ML evaluation for RobustLens on Kaggle GPUs (T4x2 or P100).
It evaluates pretrained models on Imagenette (an ImageNet subset) under various distribution shifts.

**Instructions:**
1. Open this in Kaggle.
2. Upload the `robustlens/ml` folder as a Dataset (or clone the git repo directly in the first cell).
3. Run all cells.
4. Download the `results/` folder and copy it back to your local repo.

In [ ]:
# 1. Setup Environment and Verify GPU
!pip install uv

# Assuming you cloned the repo to /kaggle/working/robustlens
# !git clone https://github.com/your-username/robustlens.git /kaggle/working/robustlens

import torch
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! This will be very slow.")

In [ ]:
# 2. Install Dependencies via UV
import os
import sys

repo_dir = "/kaggle/working/robustlens/ml"  # Adjust this path if needed
if os.path.exists(repo_dir):
    os.chdir(repo_dir)
    !uv pip install --system -e .
    sys.path.insert(0, os.path.join(repo_dir, "src"))
else:
    print(f"Could not find repo at {repo_dir}. Installing dependencies manually.")
    !uv pip install --system torch torchvision timm datasets pandas numpy tqdm

In [ ]:
# 3. Run GPU Smoke Test
import time
import torch
import torchvision.transforms as T
import timm
from tqdm.auto import tqdm

# Import our package modules
from robustlens.eval.dataset import get_imagenette_loader
from robustlens.corruptions.transforms import get_corruption_transform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Base preprocessing
base_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

print("Loading Imagenette subset...")
loader = get_imagenette_loader(batch_size=64, transform=base_transform, subset_size=1000)

model_name = "resnet50.a1_in1k"
print(f"\nLoading {model_name}...")
model = timm.create_model(model_name, pretrained=True).to(device).eval()

# Smoke test
start_time = time.time()
total_images = 0

print("Running smoke test throughput benchmark...")
with torch.no_grad():
    for images, labels, orig_labels in tqdm(loader):
        images = images.to(device)
        
        # In the real eval, we apply corruptions here, then normalize.
        # For smoke test, just normal inference.
        images = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(images)
        
        logits = model(images)
        total_images += images.size(0)

duration = time.time() - start_time
fps = total_images / duration

print(f"\n\u2705 Smoke Test Complete!")
print(f"Device: {device}")
print(f"Throughput: {fps:.2f} images/sec")
print(f"Est. time for full 500k eval: {((500000 / fps) / 3600):.2f} hours")

del model
torch.cuda.empty_cache()